# 0. 核心思想

PPO 的 Critic 太贵，DPO 需要配对数据。

GRPO 不搞绝对裁判，只要组内排名。一次性采样一组回答，向着组内最好的回答训练。

# 1. 训练过程

## 采样

对每个输入 $x$，使用当前策略（的旧版本 $\pi_{\theta_{\text{old}}}$）采样 $G$ 个回答：

$$\{y_1, y_2, \ldots, y_G\} \sim \pi_{\theta_{\text{old}}}(\cdot \mid x)$$

## 组内标准化

分别计算每个回答的奖励 $r_i = R(x, y_i)$，并进行组内标准化：

$$\hat{A}_i = \frac{r_i - \mu_r}{\sigma_r + \epsilon}$$

其中：

$$\mu_r = \frac{1}{G} \sum_{j=1}^{G} r_j, \quad \sigma_r = \sqrt{\frac{1}{G} \sum_{j=1}^{G} (r_j - \mu_r)^2}$$

- $\mu_r$：组内奖励均值——同一问题 $G$ 个回答的平均奖励，充当"基准线"（Critic 的替代品）
- $\sigma_r$：组内奖励标准差——用于归一化，消除奖励绝对尺度的影响
- $\epsilon$：数值稳定性常数（通常取 $10^{-8}$），防止除零
- $\hat{A}_i > 0$：第 $i$ 个回答比组内平均更好 → 应当强化
- $\hat{A}_i < 0$：第 $i$ 个回答比组内平均更差 → 应当抑制


**为什么组内均值可以替代 Critic？**

策略梯度只需要一个用于降低方差的 baseline，而不一定非要一个学习出来的价值网络。

Critic 的误差还可能会传播到 Policy Network，导致策略更新时的不稳定性。

但即使组内均值是一个无需训练的统计量，效果也非常依赖采样质量。重复采样也增加了计算成本。

## 目标函数

$$L_{\text{GRPO}}(\theta) = -\frac{1}{G} \sum_{i=1}^{G} \frac{1}{|y_i|} \sum_{t=1}^{|y_i|} \left[ \min\left( \rho_{i,t} \hat{A}_i, \text{clip}(\rho_{i,t}, 1-\epsilon, 1+\epsilon) \hat{A}_i \right) - \beta \cdot D_{\text{KL}}[\pi_\theta \| \pi_{\text{ref}}] \right]$$


- $\frac{1}{G}\sum_{i=1}^{G}$：对 $G$ 个回答取平均——每个回答对梯度的贡献相等
- $\frac{1}{|y_i|}\sum_{t=1}^{|y_i|}$：对第 $i$ 个回答的 token 取平均——防止长回答主导梯度（长度归一化）
- $\rho_{i,t} = \frac{\pi_{\theta}(y_{i,t}|x,y_{i,<t})}{\pi_{\theta_{\text{old}}}(y_{i,t}|x,y_{i,<t})}$：第 $i$ 个回答第 $t$ 个 token 的重要性采样比率
- $\min(\rho_{i,t}\hat{A}_i, \text{clip}(\rho_{i,t},...)\hat{A}_i)$：PPO Clip 策略损失——继承自 PPO，防止单步更新过大
- $\beta \cdot D_{\text{KL}}[\pi_\theta \|\pi_{\text{ref}}]$：KL 散度惩罚——防止策略偏离 SFT 模型太远，避免奖励黑客和语言退化。


# 3. PPO、DPO、GRPO 对比

| 维度 | PPO | DPO | GRPO |
|:---|:---|:---|:---|
| 所需模型 | Policy + Critic + Reference | Policy + Reference | Policy + Reference |
| 显存需求 | ≈ 3× 模型大小 | ≈ 2× 模型大小 | ≈ 1.5× 模型大小 |
| 训练数据 | 在线采样 + 奖励模型 | 离线偏好对 | 在线采样 + 奖励函数 |
| 优势估计 | GAE（依赖 Critic） | 无（隐式奖励差） | 组内标准化（无 Critic） |
| 更新约束 | Clip + KL | 隐式 KL（通过 β） | Clip + KL |


| 维度 | PPO | DPO | GRPO |
|:---|:---|:---|:---|
| 训练稳定性 | 中（Critic 误差传播） | 高（监督学习） | 高（无 Critic 误差） |
| 超参数数量 | 多（≥6 个） | 极少（β 1 个） | 少（≤4 个） |
| 数据效率 | 低（需在线采样） | 高（离线复用） | 中（需 G× 采样） |
| 可探索性 | 强（在线 RL） | 无（纯离线） | 强（在线 RL） |
| 能力上界 | 可超越数据 | 受限于偏好数据质量 | 可超越数据 |


# 4. 补充

**GRPO 的核心洞察是什么？它是如何替代 PPO 中的 Critic 模型的？**

GRPO 的核心洞察是——PPO 中 Critic 的本质作用只是提供一个"基准线"来将绝对奖励转为相对优势，从而降低梯度方差。对于语言模型，可以用更简单的方式获取基准线：对同一问题采样 G 个回答，用组内奖励均值作为基准线。这样就完全省去了 Critic 模型的训练和存储，显存节省约 50%。

**GRPO 的组内均值 vs PPO 的 Critic 作为基准线：**

| 对比维度 | Critic（PPO） | 组内均值（GRPO） |
|:---|:---|:---|
| **基准线形式** | 参数化价值函数：$V_\phi(s_t)$ | 非参数统计量：$\bar r=\frac{1}{G}\sum_{i=1}^{G}r_i$ |
| **优点** | 能跨样本泛化；可估计未见状态的价值；可提供 Token 级基准线，实现更细粒度的信用分配；单个 Prompt 采样较少时仍可工作 | 无需训练；没有函数逼近和追赶策略变化的问题；不会将 Critic 的估计误差传播给 Policy；实现简单，节省显存与计算资源 |
| **缺点** | 需要额外训练和计算资源；存在函数逼近误差；策略变化时价值估计容易过时；错误基准线会直接影响 Policy 的优势估计 | 依赖每组的采样数量与多样性；每个 Prompt 需要生成多条回答；通常只能提供回答级基准线，信用分配较粗；无法跨 Prompt 复用和泛化 |
| **主要瓶颈** | 长序列和稀疏终局奖励下难以准确估计中间状态价值；策略更新较快时 Critic 可能无法及时跟踪 | 当 $G$ 太小、采样多样性不足，或组内奖励全部相同时，均值估计不稳定或无法产生有效优势 |
| **本质权衡** | 用参数化价值学习和泛化能力减少采样需求 | 用同一 Prompt 的多次采样换取无 Critic 的简单、稳定训练 |

# 5. Reference

[1] [GRPO/GSPO：组内相对策略优化与奖励函数设计](https://haozhe-xing.github.io/agent_learning/zh/chapter_agentic_rl/05_grpo.html) <br>
[2] [看完能和外婆解释的PPO, DPO, GRPO强化学习](https://zhuanlan.zhihu.com/p/1984387073625593089) <br>